# Tutorial 1 — Custom PyTorch Quantized LoRA (Net class)

**Goal:** Understand the mechanics of quantized low-rank adaptation using the same conceptual architecture as your LoRA lab: token embeddings → masked mean pooling → `fc1 (100→128)` → ReLU → `fc2`.

**Important:** This notebook uses **8-bit symmetric weight-only quantization** implemented in ordinary PyTorch, **not canonical 4-bit NF4 QLoRA**. It teaches the underlying idea (frozen quantized pretrained weights + trainable A/B). For actual 4-bit NF4 QLoRA, use Tutorial 2.

**Learning objectives:** build a `Net` class, pretrain a 4-class classifier, save/reload its weights, replace `fc1` with quantized LoRA, change the head to two classes, train on sentiment examples, inspect gradients/parameter counts, compare quantization error, and save/reload adapters.

**Environment:** CPU, NVIDIA CUDA, or Apple MPS. Uses synthetic *teaching datasets*, so accuracy is not a meaningful benchmark of real AG News or IMDB performance. You can later substitute real tokenized datasets and a matching pretrained checkpoint.

## Step 0 — Setup and reproducibility
Install PyTorch in your environment first (`pip install torch`). No `bitsandbytes` is needed. The toy datasets are generated locally, so the notebook runs without internet.

In [ ]:
import math, random, copy
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

## Step 1 — Create a small reproducible dataset

We simulate two tasks with **the same token vocabulary**. The first has four labels (analogous to AG News); the second has two labels (analogous to IMDB). Tokens 1–4 act as task-relevant indicators. This lets us demonstrate checkpoint transfer without a large download. Real datasets require a shared tokenizer/vocabulary and genuine pretrained weights.

In [ ]:
VOCAB_SIZE = 1000
SEQ_LEN = 20
PAD_ID = 0

def make_examples(n, classes, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(5, VOCAB_SIZE, (n, SEQ_LEN), generator=g)
    y = torch.randint(0, classes, (n,), generator=g)
    # Insert class-specific tokens repeatedly to create a learnable signal
    x[:, :5] = (y + 1).unsqueeze(1)
    return x, y

source_x, source_y = make_examples(1200, 4, 1)
target_x, target_y = make_examples(800, 2, 2)
test_x, test_y = make_examples(200, 2, 3)
source_loader = DataLoader(TensorDataset(source_x, source_y), batch_size=64, shuffle=True)
target_loader = DataLoader(TensorDataset(target_x, target_y), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=64)
print('Source:', source_x.shape, 'Target:', target_x.shape)

## Step 2 — Define the `Net` blueprint

The embedding produces 100 features per token. Masked mean pooling reduces sequence length, giving one 100-feature vector per text. `fc1` maps 100 → 128; `fc2` maps 128 → number of classes.

In [ ]:
class Net(nn.Module):
    def __init__(self, vocab_size, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 100, padding_idx=0)
        self.fc1 = nn.Linear(100, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, token_ids):
        mask = (token_ids != 0).unsqueeze(-1)
        emb = self.embedding(token_ids)
        pooled = (emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.fc2(F.relu(self.fc1(pooled)))

source_model = Net(VOCAB_SIZE, num_classes=4).to(device)
print(source_model)
print('fc1 weight shape:', source_model.fc1.weight.shape)

## Step 3 — Pretrain and save a compatible 4-class checkpoint

In your real lab, **replace this section with your actual pretrained checkpoint**. A PyTorch `state_dict` requires the matching model class and parameter shapes.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, count = 0., 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(y)
        total_correct += (logits.argmax(1) == y).sum().item()
        count += len(y)
    return total_loss/count, total_correct/count

source_opt = torch.optim.AdamW(source_model.parameters(), lr=0.005)
for epoch in range(3):
    loss, acc = run_epoch(source_model, source_loader, source_opt)
    print(f'Source epoch {epoch+1}: loss={loss:.4f}, accuracy={acc:.3f}')

checkpoint_path = 'toy_pretrained_net.pt'
torch.save(source_model.cpu().state_dict(), checkpoint_path)
source_model.to(device)
print('Saved:', checkpoint_path)

## Step 4 — Implement LoRA matrices A and B

For `fc1` (100→128) and rank 2:

- `A`: shape `(100, 2)`, **200 trainable weights**
- `B`: shape `(2, 128)`, **256 trainable weights**
- total **456 trainable weights**, versus **12,800** in the original `fc1` weight matrix.

`std_dev = 1/sqrt(rank)` affects **initialization only**. B begins at zero, so the LoRA correction starts at zero. This notebook follows your lab's `alpha * x @ A @ B` convention (not PEFT's usual `alpha/r` scaling).

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=2, alpha=0.1):
        super().__init__()
        self.A = nn.Parameter(torch.randn(in_dim, rank) / math.sqrt(rank))
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        return self.alpha * (x @ self.A @ self.B)

lora_demo = LoRALayer(100, 128, rank=2, alpha=0.1)
print('A:', lora_demo.A.shape, 'elements:', lora_demo.A.numel())
print('B:', lora_demo.B.shape, 'elements:', lora_demo.B.numel())
print('Initial correction zero:', torch.allclose(lora_demo(torch.randn(3, 100)), torch.zeros(3, 128)))

## Step 5 — Implement frozen 8-bit weight quantization

This implementation stores the original `fc1` weights as signed `int8` plus a float scale per output channel. Each forward pass dequantizes weights to floating point for `F.linear`. Thus **storage is compressed**, but temporary floating-point weights are materialized for computation. This is a teaching implementation, not an optimized 8-bit kernel or NF4 quantizer.

Quantization: `q = round(W/scale)` and reconstruction: `W_hat = q*scale`. We register quantized weights and scales as **buffers**, not optimizer parameters.

In [ ]:
class QuantizedLinear(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        w = linear.weight.detach().float()
        scale = w.abs().amax(dim=1, keepdim=True).clamp(min=1e-8) / 127.0
        q = torch.round(w / scale).clamp(-127, 127).to(torch.int8)
        self.register_buffer('weight_int8', q)
        self.register_buffer('scale', scale)
        if linear.bias is None:
            self.bias = None
        else:
            self.register_buffer('bias', linear.bias.detach().float().clone())

    def forward(self, x):
        w_approx = self.weight_int8.float() * self.scale
        return F.linear(x, w_approx, self.bias)

q_fc1 = QuantizedLinear(source_model.fc1.cpu())
original_w = source_model.fc1.weight.detach().cpu()
reconstructed_w = q_fc1.weight_int8.float() * q_fc1.scale
print('Original dtype:', original_w.dtype, 'quantized dtype:', q_fc1.weight_int8.dtype)
print('Mean absolute quantization error:', (original_w - reconstructed_w).abs().mean().item())
source_model.to(device)

## Step 6 — Combine frozen quantized weights with LoRA

The forward calculation is `quantized_base(x) + alpha*(x@A@B)`. The quantized base is fixed; only A and B are learned.

In [ ]:
class QuantizedLinearWithLoRA(nn.Module):
    def __init__(self, linear: nn.Linear, rank=2, alpha=0.1):
        super().__init__()
        self.linear = QuantizedLinear(linear)
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

# Construct 4-class source model, load its checkpoint, then adapt to 2 classes.
model = Net(VOCAB_SIZE, num_classes=4)
model.load_state_dict(torch.load(checkpoint_path, map_location='cpu', weights_only=True))
for p in model.parameters():
    p.requires_grad = False
model.fc1 = QuantizedLinearWithLoRA(model.fc1, rank=2, alpha=0.1)
model.fc2 = nn.Linear(128, 2)  # Newly initialized trainable target-task head
model = model.to(device)
print(model)

## Step 7 — Inspect parameter counts and trainability

Check that the embedding is frozen, quantized base weights are buffers, and A/B plus the new 2-class head are trainable.

In [ ]:
for name, p in model.named_parameters():
    print(f'{name:25} {str(tuple(p.shape)):15} trainable={p.requires_grad}')
print('\nTrainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('A:', model.fc1.lora.A.numel(), 'B:', model.fc1.lora.B.numel())
print('Quantized weight dtype:', model.fc1.linear.weight_int8.dtype)
assert model.fc1.lora.A.numel() == 200
assert model.fc1.lora.B.numel() == 256

## Step 8 — Understand gradients at initialization

Because B starts at zero, A's gradient is zero on the **first backward pass**, while B can receive a nonzero gradient. After B is updated, gradients can flow to A.

In [ ]:
x_batch, y_batch = next(iter(target_loader))
x_batch, y_batch = x_batch.to(device), y_batch.to(device)
model.zero_grad()
loss = F.cross_entropy(model(x_batch), y_batch)
loss.backward()
print('Initial ||grad A||:', model.fc1.lora.A.grad.norm().item())
print('Initial ||grad B||:', model.fc1.lora.B.grad.norm().item())
model.zero_grad()

## Step 9 — Fine-tune on the 2-class task

We train LoRA A/B and the new classifier head. Learning rate and epoch count are illustrative. Since the target data are synthetic, results do not represent real sentiment performance.

In [ ]:
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=0.01)
for epoch in range(5):
    train_loss, train_acc = run_epoch(model, target_loader, optimizer)
    test_loss, test_acc = run_epoch(model, test_loader)
    print(f'Epoch {epoch+1}: train loss={train_loss:.3f}, train acc={train_acc:.3f}; test acc={test_acc:.3f}')

## Step 10 — Confirm frozen quantized weights and nonzero adapter

Quantized weights never receive gradients. The trained A/B matrices generate a nonzero correction.

In [ ]:
print('Frozen base is a buffer:', 'weight_int8' in dict(model.fc1.linear.named_buffers()))
print('Base int8 dtype:', model.fc1.linear.weight_int8.dtype)
print('LoRA correction norm:', (model.fc1.lora.A @ model.fc1.lora.B).norm().item())
print('A trainable:', model.fc1.lora.A.requires_grad, '| B trainable:', model.fc1.lora.B.requires_grad)

## Step 11 — Save and reload the adapted model

A complete checkpoint below includes quantized base buffers, LoRA weights, embedding, and new classifier head. In production you might save just adapters and the new head, alongside metadata identifying the exact base checkpoint.

In [ ]:
adapted_path = 'toy_quantized_lora_net.pt'
torch.save(model.cpu().state_dict(), adapted_path)
reloaded = Net(VOCAB_SIZE, num_classes=4)
# Rebuild the same wrapped architecture before loading.
reloaded.fc1 = QuantizedLinearWithLoRA(reloaded.fc1, rank=2, alpha=0.1)
reloaded.fc2 = nn.Linear(128, 2)
reloaded.load_state_dict(torch.load(adapted_path, map_location='cpu', weights_only=True))
reloaded.eval()
with torch.no_grad():
    prediction = reloaded(test_x[:3]).argmax(dim=1)
print('Reloaded predictions:', prediction.tolist())

## Step 12 — Exercises and troubleshooting

1. Change rank from 2 to 4. Predict A/B shapes and total trainable parameters **before running**.
2. Change alpha from 0.1 to 1.0. Does alpha itself learn? Does it change gradient magnitude?
3. Quantize `fc2` only if it is a **frozen pretrained head**; in transfer learning the newly initialized head should normally remain trainable.
4. Compare original vs quantized `fc1` outputs on the same input and measure absolute error.
5. Replace the synthetic datasets with AG News/IMDB, using a consistent tokenizer, correct label indexing, and a real pretrained checkpoint.

**Pitfalls:** A/B added *before* a global freeze would also be frozen. `state_dict` does not reconstruct custom class definitions. Quantization is lossy. Dequantization in this example creates temporary float weights, so do not interpret int8 storage as total training-memory use.

**Next:** Notebook 2 uses real 4-bit NF4 quantization and PEFT on Hugging Face.